# WHOMOLLM Project
## Geo context prompt (Version 3) 28.Jan.2026
1. Raw data
2. stay segmentation
3. reverse geocoding
4. top 5 POIs aggregation
5. semantic stay table
6. LLM prompt (daily narrative/ activity inference/ other surplus info)

## 1. Test GPU, model 

In [2]:
import os, json, re, subprocess, sys
import torch
import requests
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory/1024**3, 2))

# Check ollama is reachable
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
print("Ollama OK. Models:", [m["name"] for m in r.json().get("models",[])])

CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.06
Ollama OK. Models: ['gpt-oss:20b']


## 2. Generate prompts (3nd version with just geo context information and behaviour info, predict household size and household income level ... for all users)
Version control:
This is the basic prompt based on geocontext and behaviour info, version 3.

In [2]:
#   ---------------------------
# Version 3 (28 Jan 2026) prompt with geo context to predict features
#   ---------------------------

import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb

# ---------------------------
# Config
# ---------------------------
RANDOM_SEED = 42
N_USERS     = 5
MAX_EVENTS  = 300
HOUR_BIN    = 1
PREC        = int(os.getenv("COORD_PREC", "4"))


DATA_SP     = Path("/data/baliu/python_code/data/sp_all copy.csv")
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_features_28Jan2026.json")

# -------------------------------------------   
# Load (strings) + base cleaning
# -------------------------------------------
sp = pd.read_csv(DATA_SP, dtype=str, engine="python", on_bad_lines="skip")
print("sp shape:", sp.shape)
# print("sp columns:", sp.columns.tolist())

# Datetimes
sp["started_at"]  = pd.to_datetime(sp["started_at"], errors="coerce", utc=True)
sp["finished_at"] = pd.to_datetime(sp.get("finished_at"), errors="coerce", utc=True)

# Drop unusable
sp = sp.dropna(subset=["user_id", "started_at", "location_id"]).copy()
sp["user_id"]     = sp["user_id"].astype(str)
sp["location_id"] = sp["location_id"].astype(str)

#sp["duration_min"] = ((sp["finished_at"] - sp["started_at"]).dt.total_seconds() / 60.0).astype(float)
print(sp["duration"].head())
print(sp["duration"].dtype)

# Force numeric
sp["act_duration"] = pd.to_numeric(
    sp["act_duration"],
    errors="coerce"
)

# change it to hours
sp["act_duration_h"] = (sp["act_duration"] / 60.0).round(1)
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)

# length_km: convert to numeric, coerce errors to NaN, then divide by 1000
sp["length_km"] = pd.to_numeric(sp["length"], errors="coerce") / 1000.0
sp["length_km"] = sp["length_km"].round(1)

# Time features
sp["date"]     = sp["started_at"].dt.date
sp["dow"]      = sp["started_at"].dt.dayofweek.astype(int)        # 0=Mon
sp["hour_bin"] = (sp["started_at"].dt.hour // HOUR_BIN * HOUR_BIN).astype(int)

# ---------------------------
# Format from wkt point -> lon/lat (rounded)
# ---------------------------
WKT_POINT_RE = re.compile(
    r"POINT\s*\(\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*\)",
    re.I,
)

def extract_lonlat_from_wkt(s: str):
    m = WKT_POINT_RE.search(str(s))
    if not m: return (np.nan, np.nan)
    return float(m.group(1)), float(m.group(2))

sp["geometry"] = sp.get("geometry", pd.Series(index=sp.index)).astype(str)
lonlat = sp["geometry"].apply(extract_lonlat_from_wkt)
sp[["lon","lat"]] = pd.DataFrame(lonlat.tolist(), index=sp.index)
sp["lon"] = pd.to_numeric(sp["lon"], errors="coerce").round(PREC)
sp["lat"] = pd.to_numeric(sp["lat"], errors="coerce").round(PREC)

# Lightweight tags
sp["geometry_type"] = np.where(sp["lon"].notna() & sp["lat"].notna(), "point", "-")
if "mode" not in sp.columns: sp["mode"] = "-"

sp shape: (1152987, 10)
0    1093.0
1      39.0
2      75.0
3     164.0
4      36.0
Name: duration, dtype: object
object
0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64


In [3]:
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)
print(sp["length_km"].head())
print(sp["length_km"].dtype)

0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64
0     0.0
1    11.6
2     2.1
3     4.8
4    12.6
Name: length_km, dtype: float64
float64


In [4]:
print ("sp after cleaning:", sp.shape)
print (sp.head(5))

sp after cleaning: (862871, 18)
  id user_id                       started_at  \
0  0   AAGAF 2019-10-09 11:30:34.141000+00:00   
1  1   AAGAF 2019-10-10 06:14:49.141999+00:00   
2  2   AAGAF 2019-10-10 07:03:24.426000+00:00   
3  3   AAGAF 2019-10-10 11:10:24.605999+00:00   
4  4   AAGAF 2019-10-10 14:30:45.187999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-10 05:43:17.674999+00:00           0   Car                 0.0   
1 2019-10-10 06:53:54.841000+00:00           1   Car  11615.408548376423   
2 2019-10-10 08:18:20.864000+00:00           2   Car  2104.8558581266784   
3 2019-10-10 13:54:34.799339+00:00           2  Walk   4847.706521391238   
4 2019-10-10 15:07:07.127239+00:00           3   Car  12621.909934521313   

                                       geometry duration  act_duration  \
0  POINT (7.565219245342489 47.545616372908086)   1093.0        1093.0   
1  POINT (7.5637597959179645 47.54794767256367)     39.0          71

In [5]:
# ---------------------------
# Sample 1 week per user
# ---------------------------
def sample_one_week_per_user(sp, seed=42):
    rng = np.random.default_rng(seed)
    sp = sp.copy()
    sp["date"] = pd.to_datetime(sp["date"])

    out = []
    for uid, df_u in sp.groupby("user_id"):
        days = (
            df_u[["date"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(days) < 7:
            continue
        days["delta"] = days["date"].diff().dt.days.fillna(1).astype(int)
        days["block"] = (days["delta"] >1).cumsum()

        valid_blocks = (
            days.groupby("block")
            .filter(lambda x: len(x) >=7)
        )

        if valid_blocks.empty:
            continue
        candidate_starts = []
        for _, g in valid_blocks.groupby("block"):
            block_dates = g["date"].sort_values().reset_index(drop=True)
            for i in range(len(block_dates) - 6):
                candidate_starts.append(g.iloc[i]["date"])

        start_date = rng.choice(candidate_starts)
        week_dates = pd.date_range(start=start_date, periods=7, freq= 'D')

        out.append(
            df_u[df_u["date"].isin(week_dates)]

        )
       
    return pd.concat(out, ignore_index=True)

sp_sampled2 = sample_one_week_per_user(sp, seed=RANDOM_SEED)
print("sp_sampled2 shape:", sp_sampled2.shape)
print(sp_sampled2.head(2))

# ---------------------------
# save sampled data
# ---------------------------
SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_1week_28Jan2026.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)
print("Saved sp_sampled2_1week_28Jan2026 to", SP_SAMPLED2_OUT)  

sp_sampled2 shape: (60738, 18)
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   

                       finished_at location_id  mode             length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus  3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram  8232.492568529495   

                                       geometry duration  act_duration  \
0  POINT (7.564359990583688 47.546459921667946)   1097.0        6456.0   
1   POINT (7.603269084785703 47.58100341208616)    116.0         275.0   

   act_duration_h  length_km       date  dow  hour_bin     lon      lat  \
0           107.6        3.8 2019-10-22    1         9  7.5644  47.5465   
1             4.6        8.2 2019-10-23    2         6  7.6033  47.5810   

  geometry_type  
0         point  
1         point  
Saved sp_sampled2_1week_28Jan2026 to /data/baliu/python_code/data/sp_sampled2_1week_28Jan

#### Load location name and POIs top 5

In [1]:
# 1. Load toponym data from shepefiles
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb
import geopandas as gpd
import fiona
# read parquet file
import pyarrow.parquet as pq
 
CACHE_DIR   = Path("/data/baliu/python_code/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NOM_CACHE_PATH = CACHE_DIR / "nominatim_cache.parquet"
nom = pd.read_parquet(NOM_CACHE_PATH)
print("Nominatim cache loaded:", nom.shape)
print("Nominatim columns:", nom.columns.tolist())
nom.head(30)
groupby_category = nom.groupby('city').size()
print(groupby_category) 

Nominatim cache loaded: (370983, 6)
Nominatim columns: ['lon', 'lat', 'road', 'neighbourhood', 'city', 'postcode']
city
A Coruña                 2
A Illa de Arousa         2
A Pobra do Caramiñal     7
A Seara                  1
Aachen                   2
                        ..
دهستان سروستان           1
دهستان فجر               3
دهستان کرکس              1
บ้านอ่าวน้ำเมา           1
เทศบาลนครเกาะสมุย       55
Length: 7400, dtype: int64


In [2]:
from notebook_utils.geocontext import attach_nominatim, clean_nominatim_fields

sp_sampled2 = attach_nominatim(sp_sampled2, nom)
sp_sampled2 = clean_nominatim_fields(sp_sampled2)
print("sp_sampled2 after attaching nominatim:", sp_sampled2.shape)
print("sp_sampled2 columns:", sp_sampled2.columns.tolist())
print(sp_sampled2.head(10))


NameError: name 'sp_sampled2' is not defined

#### Build token with geocontext info

In [9]:
from pathlib import Path
from notebook_utils.geocontext import load_poi_frame

POI_PATH = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")
poi = load_poi_frame(POI_PATH)
pois = poi
poi.head()


,id,code,name,category,geometry
0,0,3100,Temple de Saint-Jean,Civic,POINT (2498816.948 1117839.539)
1,1,3100,Kapelle Oberes Heiliges Kreuz,Civic,POINT (2691857.357 1192677.631)
2,2,3102,Kirche St. Johannes,Civic,POINT (2599659.594 1202970.041)
3,3,3102,Paroisse catholique,Civic,POINT (2501879.911 1126418.086)
4,4,3300,Mosquée du Petit-Saconnex,Civic,POINT (2498411.634 1119984.893)


In [10]:
poi = poi.copy()
print(poi["category"].value_counts())
print(f"poi crs: {poi.crs}")


category
Others            158579
Unknown           114894
Entertainment     106734
Shopping           54634
Residential        45556
Transportation     40997
Services            9525
Schools             2904
Civic                780
Name: count, dtype: int64
poi crs: EPSG:2056


In [11]:
from notebook_utils.geocontext import load_poi_frame

def load_pois_once():
    global poi
    if poi is None:
        print("Loading POIs into memory...")
        poi = load_poi_frame(POI_PATH)
    return poi

print(poi.head(2))


   id  code                           name category  \
0   0  3100           Temple de Saint-Jean    Civic   
1   1  3100  Kapelle Oberes Heiliges Kreuz    Civic   

                          geometry  
0  POINT (2498816.948 1117839.539)  
1  POINT (2691857.357 1192677.631)  


In [13]:
from notebook_utils.geocontext import build_location_gdf

loc = build_location_gdf(sp_sampled2)


### update the table with places name and pois

update

In [24]:
# update: 
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on POIs ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 5) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def clean_text_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        dist = r["addr_poi_dist_km"]
        direction = r["direction"].lower()
        
       
        category = clean_text_part(r.get("category")) or "point of interest"
        name = clean_text_part(r.get("name"))

        if name:
            poi_desc = f"{category} {name}"
        else:
            poi_desc = f"{category}"
        out.append(
            f"{dist} km to the {direction}: {poi_desc}"
        )
    return out

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_1week_29Jan2026_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

count    116301.000000
mean        288.675402
std        1458.250814
min           0.000000
25%           0.042000
50%           0.101000
75%           0.262000
max       14837.215000
Name: addr_poi_dist_km, dtype: float64
Saved sp_sampled2 to /data/baliu/python_code/data/sp_sampled2_v3_1week_29Jan2026_with_geocontext.csv
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.143616000176

In [ ]:
# update: 
import numpy as np

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on pois ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 3) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        name = r["name"] if pd.notna(r["name"]) else "unknown place"
        category = r["category"] if pd.notna(r["category"]) else "point of interest"
        out.append(
            f"{r['addr_poi_dist_km']} km to the {r['direction'].lower()}: {category} {name}"
        )
    return "; ".join(out)

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

print("nearby_places in columns",
      "nearby_places" in sp_sampled2.columns)

print(
    sp_sampled2["nearby_places"]
    .notna()
    .value_counts(dropna=False)
)

print(sp_sampled2[["location_id", "nearby_places"]].head(5))


from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

count    226778.000000
mean        296.192856
std        1476.196361
min           0.000000
25%           0.059000
50%           0.133000
75%           0.355000
max       14837.298000
Name: addr_poi_dist_km, dtype: float64
nearby_places in columns? True
nearby_places
True     60319
False      419
Name: count, dtype: int64
  location_id                                      nearby_places
0          10  0.004 km to the east: Shopping Tabaklädeli Neu...
1          16  0.086 km to the south-east: Shopping unknown p...
2          17  0.023 km to the west: Entertainment unknown pl...
3          10  0.004 km to the east: Shopping Tabaklädeli Neu...
4           4  1.748 km to the south-east: Entertainment Ökos...
Saved sp_sampled1 to /data/baliu/python_code/data/sp_sampled1_v3_with_geocontext.csv
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3

same as above, upgraded 

In [25]:
from notebook_utils.geocontext import print_poi_feature_summary

print_poi_feature_summary(sp_sampled2)



Final sp_sampled2 with POI features:
Columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode', 'nearby_places_x', 'nearby_places_y', 'nearby_places']

Sample rows:
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816

Clean the prompts 

## Baseline_ Knn  and Random forest model 

 We have installed scikit-learn and upgrade this package and xgboost, then i just deleted the records

In [ ]:
"""
Household income prediction from mobility data
Using KNN and random forest classifiers
"""

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. Data loading and preprocessing 
# ============================================================================


#SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled12_with_geocontext.csv")
#sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)


def load_data(trace_file='/data/baliu/python_code/data/sp_sampled12_with_geocontext.csv', survey_file='/data/baliu/python_code/data/introSurvey_complete.csv'):
    # trajectory data
    trace_df = pd.read_csv(trace_file)
    print(f"Trace data shape: {trace_df.shape}")
    
    # Load survey data (ground truth)
    survey_df = pd.read_csv(survey_file)
    print(f"Survey data shape: {survey_df.shape}")
    
    return trace_df, survey_df


def categorize_income(income_value):
    """
    Categorize household income from survey text labels.
    """

    if pd.isna(income_value):
        return None

    s = str(income_value).strip().lower()

    if s == "prefer not to say":
        return None

    if "4 000 chf or less" in s or "4000 chf or less" in s:
        return "0_<4000"

    if "4 001 - 8 000 chf" in s:
        return "1_4001-8000"

    if "8 001 - 12 000 chf" in s:
        return "2_8001-12000"

    if "12 001 - 16 000 chf" in s:
        return "3_12001-16000"

    if "more than 16 000 chf" in s:
        return "4_>16001"

    return None


def preprocess_survey_data(survey_df):
    """Preprocess survey data and create income categories"""
    print("\nPreprocessing survey data...")

    survey_df['income_category'] = survey_df['income'].apply(categorize_income)

    # Drop "Prefer not to say"
    survey_df = survey_df.dropna(subset=['income_category'])

    survey_df = survey_df[['participant_ID', 'income', 'income_category']]

    print("Income category distribution:")
    print(survey_df['income_category'].value_counts().sort_index())

    return survey_df


# ============================================================================
# 2. Feature engineering from mobility traces 
# ============================================================================

# temporal features: trip count, time of day, day of week, peak hours on the weekdays, weekend vs weekday, time diversity
def extract_temporal_features(group):
    """Extract time-based features from mobility traces"""
    
    # Convert to datetime if not already
    if 'start_at' in group.columns:
        group['started_at'] = pd.to_datetime(group['start_at'])
        group['finished_at'] = pd.to_datetime(group['end_at'])
    
    features = {}
    
    # Trip count features
    features['total_trips'] = len(group)
    


    # Temporal features
    if 'started_at' in group.columns:
        group['hour'] = group['started_at'].dt.hour
        group['day_of_week'] = group['started_at'].dt.dayofweek
        
        # Peak hour trips (6-9 am, 4-7 pm) on weekdays, just in Switzerland
        features['morning_peak_trips'] = ((group['hour'] >= 6) & (group['hour'] <= 9) & (group['day_of_week'] < 5)).sum()
        features['evening_peak_trips'] = ((group['hour'] >= 16) & (group['hour'] <= 19) & (group['day_of_week'] < 5)).sum()
        features['peak_trip_ratio'] = (features['morning_peak_trips'] + features['evening_peak_trips']) / max(features['total_trips'], 1)
        
        # Weekend vs weekday
        features['weekday_trips'] = (group['day_of_week'] < 5).sum()
        features['weekend_trips'] = (group['day_of_week'] >= 5).sum()
        features['weekend_trip_ratio'] = features['weekend_trips'] / max(features['total_trips'], 1)
        
        # Time diversity
        features['unique_hours'] = group['hour'].nunique()
        features['unique_days'] = group['started_at'].dt.date.nunique()
    
    return features


# Need to update the proper coordinate parsing and real radius of gyration calculation
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def calculate_radius_of_gyration(group):
    if 'lon' in group.columns and 'lat' in group.columns and len(group) > 1:
        lat_mean = group['lat'].mean()
        lon_mean = group['lon'].mean()
        distances = group.apply(lambda row: haversine_distance(
            row['lat'], row['lon'], lat_mean, lon_mean), axis=1)
        return np.sqrt((distances**2).mean())
    return 0

# spatial features: distance stats, location diversity, radius of gyration
def extract_spatial_features(group):
    """Extract location-based features from mobility traces"""
    
    features = {}
    
    # Distance features
    if 'distance' in group.columns:
        features['total_distance'] = group['distance'].sum()
        features['avg_distance'] = group['distance'].mean()
        features['max_distance'] = group['distance'].max()
        features['min_distance'] = group['distance'].min()
        features['std_distance'] = group['distance'].std() if len(group) > 1 else 0
        features['distance_range'] = features['max_distance'] - features['min_distance']
    
    # Location diversity, we have location_id to compute the unique locations visited and their entropy
    if 'location_id' in group.columns:
        features['unique_locations'] = group['location_id'].nunique()
        features['location_entropy'] = calculate_entropy(group['location_id'].value_counts())
    
    # Radius of gyration (measure of travel area)
    if 'geometry' in group.columns and 'length_km' in group.columns:
        features['radius_of_gyration'] = calculate_radius_of_gyration(group)
    
    return features


def extract_mode_features(group):
    """Extract transportation mode features"""
    
    features = {}
    
    if 'mode' in group.columns:
        # Mode distribution
        mode_counts = group['mode'].value_counts()
        total_trips = len(group)
        
        # Get unique modes
        for mode in group['mode'].unique():
            features[f'mode_{mode}_count'] = mode_counts.get(mode, 0)
            features[f'mode_{mode}_ratio'] = mode_counts.get(mode, 0) / total_trips
        
        # Mode diversity
        features['mode_diversity'] = group['mode'].nunique()
        features['mode_entropy'] = calculate_entropy(mode_counts)
        
        # Dominant mode
        features['dominant_mode_ratio'] = mode_counts.max() / total_trips
    
    return features


def calculate_entropy(value_counts):
    """Calculate Shannon entropy for diversity measure"""
    probabilities = value_counts / value_counts.sum()
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
    return entropy


def calculate_radius_of_gyration(group):
    """
    Calculate radius of gyration - measure of travel area
    Simplified version using distance variance
    """
    if 'distance' in group.columns and len(group) > 1:
        return np.sqrt(group['distance'].var())
    return 0


def build_features_per_user(trace_df):
    """Build comprehensive feature set for each user"""
    print("\nBuilding features from mobility traces...")
    
    all_features = []
    
    # Group by user_id
    for user_id, user_group in trace_df.groupby('user_id'):
        user_features = {'user_id': user_id}
        
        # Extract different types of features
        temporal_features = extract_temporal_features(user_group)
        spatial_features = extract_spatial_features(user_group)
        mode_features = extract_mode_features(user_group)
        
        # Combine all features
        user_features.update(temporal_features)
        user_features.update(spatial_features)
        user_features.update(mode_features)
        
        # Additional derived features
        if 'total_trips' in user_features and 'unique_days' in user_features:
            user_features['trips_per_day'] = user_features['total_trips'] / max(user_features['unique_days'], 1)
        
        if 'total_distance' in user_features and 'unique_days' in user_features:
            user_features['distance_per_day'] = user_features['total_distance'] / max(user_features['unique_days'], 1)
        
        all_features.append(user_features)
    
    features_df = pd.DataFrame(all_features).fillna(0)
    
    print(f"Feature matrix shape: {features_df.shape}")
    print(f"Number of features: {len(features_df.columns) - 1}")  # -1 for user_id
    
    return features_df


# ============================================================================
# 3. Model training and evaluation 
# ============================================================================

def prepare_train_test_data(features_df, survey_df, test_size=0.2, random_state=42):
    """Merge features with labels and split into train/test sets"""
    print("\nPreparing training and test data...")
    
    # Merge features with survey data
    # Match user_id with participant_ID
    merged_df = features_df.merge(
        survey_df, 
        left_on='user_id', 
        right_on='participant_ID', 
        how='inner'
    )
    
    print(f"Merged data shape: {merged_df.shape}")
    print(f"Number of users with both trace and income data: {len(merged_df)}")
    print("Merged columns:", merged_df.columns.tolist())
    print(merged_df.head(2))
    
    # Separate features and target
    feature_cols = [col for col in merged_df.columns 
                   if col not in ['user_id', 'participant_ID', 'income', 'income_category']]
    
    X = merged_df[feature_cols]
    y = merged_df['income_category']
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"\nFeature columns ({len(feature_cols)}):")
    print(feature_cols[:10], "...")  # Print first 10
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state, stratify=y_encoded
    )
    
    print(f"\nTrain set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    
    return X_train, X_test, y_train, y_test, label_encoder, feature_cols


def train_knn_model(X_train, y_train, X_test, y_test):
    """Train and evaluate KNN classifier"""
    print("\n" + "="*70)
    print("Training knn model")
    print("="*70)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Grid search for best k
    param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11, 21],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
    
    knn = KNeighborsClassifier()
    grid_search = GridSearchCV(
        knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    print("\nPerforming grid search for best parameters...")
    grid_search.fit(X_train_scaled, y_train)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    # Best model
    best_knn = grid_search.best_estimator_
    
    # Predictions
    y_pred_train = best_knn.predict(X_train_scaled)
    y_pred_test = best_knn.predict(X_test_scaled)
    
    # Evaluation
    print("\n" + "-"*70)
    print("knn model performance")
    print("-"*70)
    print(f"Training acuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
    
    return best_knn, scaler


def train_random_forest_model(X_train, y_train, X_test, y_test, feature_cols):
    """Train and evaluate random forest classifier"""
    print("\n" + "="*70)
    print("Training random forest model")
    print("="*70)
    
    # Grid search for best parameters
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, 10],
        'min_samples_split': [5, 10, 20],
        'min_samples_leaf': [2, 4, 8],
        'max_features': ['sqrt', 'log2']
    }
    
    rf = RandomForestClassifier(random_state=42, class_weight='balanced') # Handle class imbalance
    grid_search = GridSearchCV(
        rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    print("\nPerforming grid search for best parameters...")
    grid_search.fit(X_train, y_train)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    # Best model
    best_rf = grid_search.best_estimator_
    
    # Predictions
    y_pred_train = best_rf.predict(X_train)
    y_pred_test = best_rf.predict(X_test)
    
    # Evaluation
    print("\n" + "-"*70)
    print("Random forest model performance")
    print("-"*70)
    print(f"Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n" + "-"*70)
    print("Top 15 feature importances")
    print("-"*70)
    print(feature_importance.head(15).to_string(index=False))
    
    return best_rf, feature_importance


def evaluate_models(models_dict, X_test, y_test, label_encoder):
    """Comprehensive evaluation of all models"""
    print("\n" + "="*70)
    print("Comprehensive model evaluation")
    print("="*70)
    
    for model_name, model_info in models_dict.items():
        print(f"\n{'='*70}")
        print(f"{model_name.upper()}")
        print(f"{'='*70}")
        
        model = model_info['model']
        
        # Get test data (scaled for KNN)
        if 'scaler' in model_info:
            X_test_processed = model_info['scaler'].transform(X_test)
        else:
            X_test_processed = X_test
        
        # Predictions
        y_pred = model.predict(X_test_processed)
        
        # Classification report
        print("\nClassification Report:")
        print(classification_report(
            y_test, y_pred, 
            target_names=label_encoder.classes_,
            digits=4
        ))
        
        # Confusion matrix
        print("\nConfusion Matrix:")
        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(
            cm, 
            index=label_encoder.classes_, 
            columns=label_encoder.classes_
        )
        print(cm_df)
        
        # Overall accuracy
        accuracy = accuracy_score(y_test, y_pred)
        print(f"\nOverall Test Accuracy: {accuracy:.4f}")


# ============================================================================
# 4. Main pipeline
# ============================================================================

def main():
    """Main pipeline"""
    
    print("="*70)
    print("Household income prediciton for mobility traces based on random forest and knn")
    print("="*70)
    
    # 1. Load data
    trace_df, survey_df = load_data()
    
    # 2. Preprocess survey data
    survey_df = preprocess_survey_data(survey_df)
    
    trace_df['started_at'] = pd.to_datetime(trace_df['started_at'], errors='coerce')
    trace_df['finished_at']   = pd.to_datetime(trace_df['finished_at'], errors='coerce')
    trace_df = trace_df.dropna(subset=['started_at', 'finished_at'])

    # 3. Build features from mobility traces
    features_df = build_features_per_user(trace_df)
    
    # 4. Prepare train/test data
    X_train, X_test, y_train, y_test, label_encoder, feature_cols = prepare_train_test_data(
        features_df, survey_df
    )
    
    # 5. Train KNN model
    knn_model, knn_scaler = train_knn_model(X_train, y_train, X_test, y_test)
    
    # 6. Train random forest model
    rf_model, feature_importance = train_random_forest_model(
        X_train, y_train, X_test, y_test, feature_cols
    )
    
    # 7. Comprehensive evaluation
    models_dict = {
        'KNN': {'model': knn_model, 'scaler': knn_scaler},
        'Random forest': {'model': rf_model}
    }
    
    evaluate_models(models_dict, X_test, y_test, label_encoder)
    
    # 8. Save feature importance
    feature_importance.to_csv('/data/baliu/python_code/data/feature_importance.csv', index=False)
    
    # 9. Save models
    import pickle
    
    with open('/data/baliu/python_code/data/knn_model.pkl', 'wb') as f:
        pickle.dump({'model': knn_model, 'scaler': knn_scaler}, f)
    
    with open('/data/baliu/python_code/data/rf_model.pkl', 'wb') as f:
        pickle.dump(rf_model, f)
    
    with open('/data/baliu/python_code/data/label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)
    
    print("✓ Models saved successfully")

if __name__ == "__main__":
    main()

Household income prediciton for mobility traces based on random forest and knn
Trace data shape: (16714, 23)
Survey data shape: (90909, 124)

Preprocessing survey data...
Income category distribution:
income_category
0_<4000          2618
1_4001-8000      6454
2_8001-12000     5285
3_12001-16000    2575
4_>16001         1700
Name: count, dtype: int64

Building features from mobility traces...
Feature matrix shape: (2113, 31)
Number of features: 30

Preparing training and test data...
Merged data shape: (1534, 34)
Number of users with both trace and income data: 1534
Merged columns: ['user_id', 'total_trips', 'morning_peak_trips', 'evening_peak_trips', 'peak_trip_ratio', 'weekday_trips', 'weekend_trips', 'weekend_trip_ratio', 'unique_hours', 'unique_days', 'unique_locations', 'location_entropy', 'radius_of_gyration', 'mode_Car_count', 'mode_Car_ratio', 'mode_Tram_count', 'mode_Tram_ratio', 'mode_Walk_count', 'mode_Walk_ratio', 'mode_diversity', 'mode_entropy', 'dominant_mode_ratio', 'tr

## SVM and XGBoost models based on the 1 week data

In [ ]:
"""
Household income prediction from mobility data
Using SVM (RBF kernel) and XGBoost classifiers

Same data pipeline and feature engineering as the RF/KNN version.
All previous fixes carried over:
    - class_weight='balanced' (SVM) / sample_weight (XGBoost)
    - f1_macro scoring in GridSearchCV
    - proper radius of gyration with haversine
    - correct geometry parsing
    - 3-class collapse option
    - single datetime conversion
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================
USE_3_CLASS = False   # True  →  collapse into Low / Mid / High


# ============================================================================
# 1. Data loading and preprocessing
# ============================================================================

def load_data(trace_file='/data/baliu/python_code/data/sp_sampled2_v3_with_geocontext.csv',
              survey_file='/data/baliu/python_code/data/introSurvey_complete.csv'):
    trace_df  = pd.read_csv(trace_file)
    survey_df = pd.read_csv(survey_file)
    print(f"Trace data shape:  {trace_df.shape}")
    print(f"Survey data shape: {survey_df.shape}")
    return trace_df, survey_df


def categorize_income(income_value):
    """Map survey text labels → ordered income bins."""
    if pd.isna(income_value):
        return None
    s = str(income_value).strip().lower()
    if s == "prefer not to say":
        return None
    if "4 000 chf or less" in s or "4000 chf or less" in s:
        return "0_<4000"
    if "4 001 - 8 000 chf" in s:
        return "1_4001-8000"
    if "8 001 - 12 000 chf" in s:
        return "2_8001-12000"
    if "12 001 - 16 000 chf" in s:
        return "3_12001-16000"
    if "more than 16 000 chf" in s:
        return "4_>16001"
    return None


def collapse_to_3_class(cat):
    """
    Collapse 5 bins → 3 when minority classes are too small.
        Low  = 0_<4000  +  1_4001-8000
        Mid  = 2_8001-12000
        High = 3_12001-16000  +  4_>16001
    """
    if cat in ("0_<4000", "1_4001-8000"):
        return "0_Low"
    if cat == "2_8001-12000":
        return "1_Mid"
    if cat in ("3_12001-16000", "4_>16001"):
        return "2_High"
    return None


def preprocess_survey_data(survey_df):
    """Preprocess survey data and create income categories."""
    print("\nPreprocessing survey data...")
    survey_df = survey_df.copy()
    survey_df['income_category'] = survey_df['income'].apply(categorize_income)
    survey_df = survey_df.dropna(subset=['income_category'])

    if USE_3_CLASS:
        survey_df['income_category'] = survey_df['income_category'].apply(collapse_to_3_class)
        survey_df = survey_df.dropna(subset=['income_category'])
        print("  → 3-class mode enabled (Low / Mid / High)")

    survey_df = survey_df[['participant_ID', 'income', 'income_category']]
    print("Income category distribution:")
    print(survey_df['income_category'].value_counts().sort_index())
    return survey_df


# ============================================================================
# 2. Feature engineering  (identical to the RF/KNN version)
# ============================================================================

def parse_geometry_to_coords(geometry_series):
    """
    Extract (lat, lon) from a geometry column.
    Handles:
        - "POINT (lon lat)"   (WKT)
        - "(lat, lon)"        (tuple string)
    """
    coords = pd.DataFrame(index=geometry_series.index, columns=['lat', 'lon'], dtype=float)
    for idx, val in geometry_series.items():
        if pd.isna(val):
            continue
        s = str(val).strip()
        if s.upper().startswith("POINT"):
            inner = s.split("(")[-1].rstrip(")")
            parts = inner.strip().split()
            if len(parts) == 2:
                coords.at[idx, 'lon'] = float(parts[0])
                coords.at[idx, 'lat'] = float(parts[1])
        else:
            cleaned = s.strip("()")
            parts   = cleaned.split(",")
            if len(parts) == 2:
                coords.at[idx, 'lat'] = float(parts[0].strip())
                coords.at[idx, 'lon'] = float(parts[1].strip())
    return coords


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine → km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def calculate_radius_of_gyration(coords_df):
    """Rog = sqrt( mean( d(point_i, centre)^2 ) )  using haversine."""
    valid = coords_df.dropna(subset=['lat', 'lon'])
    if len(valid) < 2:
        return 0.0
    centre_lat, centre_lon = valid['lat'].mean(), valid['lon'].mean()
    d = haversine_km(valid['lat'].values, valid['lon'].values, centre_lat, centre_lon)
    return np.sqrt((d**2).mean())


def calculate_entropy(value_counts):
    """Shannon entropy."""
    p = value_counts / value_counts.sum()
    return -np.sum(p * np.log2(p + 1e-10))


# ---------- temporal ----------

def extract_temporal_features(group):
    features = {}
    features['total_trips'] = len(group)

    if 'started_at' in group.columns:
        group = group.copy()
        group['hour']        = group['started_at'].dt.hour
        group['day_of_week'] = group['started_at'].dt.dayofweek

        features['morning_peak_trips'] = ((group['hour'] >= 6)  & (group['hour'] <= 9)  & (group['day_of_week'] < 5)).sum()
        features['evening_peak_trips'] = ((group['hour'] >= 16) & (group['hour'] <= 19) & (group['day_of_week'] < 5)).sum()
        features['peak_trip_ratio']    = (features['morning_peak_trips'] + features['evening_peak_trips']) / max(features['total_trips'], 1)

        features['weekday_trips']      = (group['day_of_week'] < 5).sum()
        features['weekend_trips']      = (group['day_of_week'] >= 5).sum()
        features['weekend_trip_ratio'] = features['weekend_trips'] / max(features['total_trips'], 1)

        features['unique_hours'] = group['hour'].nunique()
        features['unique_days']  = group['started_at'].dt.date.nunique()

    return features


# ---------- spatial ----------

def extract_spatial_features(group):
    features = {}

    if 'distance' in group.columns:
        features['total_distance'] = group['distance'].sum()
        features['avg_distance']   = group['distance'].mean()
        features['max_distance']   = group['distance'].max()
        features['min_distance']   = group['distance'].min()
        features['std_distance']   = group['distance'].std() if len(group) > 1 else 0
        features['distance_range'] = features['max_distance'] - features['min_distance']

    if 'geometry' in group.columns:
        coords       = parse_geometry_to_coords(group['geometry'])
        valid_coords = coords.dropna(subset=['lat', 'lon'])

        if len(valid_coords) > 0:
            rounded      = valid_coords.copy()
            rounded['lat_r'] = rounded['lat'].round(3)
            rounded['lon_r'] = rounded['lon'].round(3)
            location_keys    = rounded[['lat_r', 'lon_r']].apply(tuple, axis=1)

            features['unique_locations']   = location_keys.nunique()
            features['location_entropy']   = calculate_entropy(location_keys.value_counts())
            features['radius_of_gyration'] = calculate_radius_of_gyration(valid_coords)
        else:
            features['unique_locations']   = 0
            features['location_entropy']   = 0.0
            features['radius_of_gyration'] = 0.0

    elif 'lat' in group.columns and 'lon' in group.columns:
        valid_coords = group[['lat', 'lon']].dropna()
        if len(valid_coords) > 0:
            rounded      = valid_coords.copy()
            rounded['lat_r'] = rounded['lat'].round(3)
            rounded['lon_r'] = rounded['lon'].round(3)
            location_keys    = rounded[['lat_r', 'lon_r']].apply(tuple, axis=1)

            features['unique_locations']   = location_keys.nunique()
            features['location_entropy']   = calculate_entropy(location_keys.value_counts())
            features['radius_of_gyration'] = calculate_radius_of_gyration(valid_coords)
        else:
            features['unique_locations']   = 0
            features['location_entropy']   = 0.0
            features['radius_of_gyration'] = 0.0

    return features


# ---------- mode ----------

def extract_mode_features(group):
    features = {}
    if 'mode' in group.columns:
        mode_counts = group['mode'].value_counts()
        total       = len(group)
        for mode in group['mode'].unique():
            features[f'mode_{mode}_count'] = mode_counts.get(mode, 0)
            features[f'mode_{mode}_ratio'] = mode_counts.get(mode, 0) / total
        features['mode_diversity']      = group['mode'].nunique()
        features['mode_entropy']        = calculate_entropy(mode_counts)
        features['dominant_mode_ratio'] = mode_counts.max() / total
    return features


# ---------- per-user aggregation ----------

def build_features_per_user(trace_df):
    print("\nBuilding features from mobility traces...")
    all_features = []

    for user_id, grp in trace_df.groupby('user_id'):
        feat = {'user_id': user_id}
        feat.update(extract_temporal_features(grp))
        feat.update(extract_spatial_features(grp))
        feat.update(extract_mode_features(grp))

        if 'total_trips' in feat and 'unique_days' in feat:
            feat['trips_per_day']    = feat['total_trips']    / max(feat['unique_days'], 1)
        if 'total_distance' in feat and 'unique_days' in feat:
            feat['distance_per_day'] = feat['total_distance'] / max(feat['unique_days'], 1)

        all_features.append(feat)

    features_df = pd.DataFrame(all_features).fillna(0)
    print(f"Feature matrix shape: {features_df.shape}")
    print(f"Number of features:   {len(features_df.columns) - 1}")
    return features_df


# ============================================================================
# 3. Train / test prep
# ============================================================================

def prepare_train_test_data(features_df, survey_df, test_size=0.2, random_state=42):
    print("\nPreparing training and test data...")

    merged = features_df.merge(survey_df, left_on='user_id', right_on='participant_ID', how='inner')
    print(f"Merged shape: {merged.shape}   |   matched users: {len(merged)}")

    feature_cols = [c for c in merged.columns
                    if c not in ['user_id', 'participant_ID', 'income', 'income_category']]

    X = merged[feature_cols]
    y = merged['income_category']

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    print(f"\nFeature columns ({len(feature_cols)}):")
    print(feature_cols[:10], "..." if len(feature_cols) > 10 else "")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=test_size, random_state=random_state, stratify=y_enc
    )
    print(f"Train: {len(X_train)}   Test: {len(X_test)}")
    return X_train, X_test, y_train, y_test, le, feature_cols


# ============================================================================
# 4. SVM  (RBF kernel, class_weight='balanced', f1_macro)
# ============================================================================
# WHY these choices:
#   - RBF kernel   → non-linear decision boundaries; mobility features are not linearly separable
#   - probability=True → needed if you later want predict_proba (e.g. for calibration plots)
#   - class_weight='balanced' → same imbalance fix as RF
#   - StandardScaler → SVM is distance-based, so scale is critical
#   - Grid over C and gamma → C controls margin width; gamma controls RBF radius
# ============================================================================

def train_svm_model(X_train, y_train, X_test, y_test):
    print("\n" + "=" * 70)
    print("Training SVM model  (RBF kernel)")
    print("=" * 70)

    # SVM is very sensitive to feature scale
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # C     : regularisation  — small C = wider margin (more bias), large C = narrow margin (more variance)
    # gamma : RBF width       — 'scale' = 1/(n_features * X.var()), 'auto' = 1/n_features
    #         explicit floats give finer control
    param_grid = {
        'C':     [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.01, 0.001],
        'kernel': ['rbf']                          # kept as list for easy swap to 'poly' / 'linear'
    }

    svm = SVC(class_weight='balanced', probability=True, random_state=42)

    grid = GridSearchCV(
        svm, param_grid,
        cv=5, scoring='f1_macro', n_jobs=-1, verbose=1
    )
    print("\nGrid search (scoring = f1_macro, class_weight = balanced)...")
    grid.fit(X_train_s, y_train)

    print(f"\nBest params:      {grid.best_params_}")
    print(f"Best CV f1_macro: {grid.best_score_:.4f}")

    best_svm = grid.best_estimator_

    y_pred_train = best_svm.predict(X_train_s)
    y_pred_test  = best_svm.predict(X_test_s)

    print("-" * 70)
    print(f"Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test accuracy:     {accuracy_score(y_test,  y_pred_test):.4f}")

    return best_svm, scaler


# ============================================================================
# 5. XGBoost  (multi-class, sample_weight for imbalance, early stopping)
# ============================================================================
# WHY these choices:
#   - objective='multi:softprob'  → native multi-class; outputs class probabilities
#   - eval_metric='mlogloss'      → standard for multi-class; works with early stopping
#   - sample_weight             → XGBoost has no class_weight param; the equivalent is
#                                  passing per-sample weights = (total / (n_classes * class_count))
#   - early_stopping_rounds=20  → stops if val loss hasn't improved in 20 rounds → prevents overfitting
#   - eval_set on validation    → required for early stopping
#   - max_depth 3-6             → shallow trees; XGBoost already ensembles many of them
#   - learning_rate 0.05-0.1    → small steps; let n_estimators (boosting rounds) do the work
#   - subsample / colsample     → row / column sampling per tree → regularisation
# ============================================================================

def compute_sample_weights(y_train):
    """
    Mirror sklearn's class_weight='balanced':
        w_i  =  n_samples  /  (n_classes * n_samples_in_class_i)
    """
    classes, counts = np.unique(y_train, return_counts=True)
    n_samples       = len(y_train)
    n_classes       = len(classes)
    weight_map      = dict(zip(classes, n_samples / (n_classes * counts)))
    return np.array([weight_map[y] for y in y_train])


def train_xgboost_model(X_train, y_train, X_test, y_test, feature_cols):
    print("\n" + "=" * 70)
    print("Training XGBoost model")
    print("=" * 70)

    n_classes = len(np.unique(y_train))
    sample_weights = compute_sample_weights(y_train)

    # --- manual grid search with early stopping ---
    # GridSearchCV doesn't support early_stopping_rounds cleanly, so we
    # iterate over the important hyper-params ourselves and track the best.

    param_candidates = [
        # (max_depth, learning_rate, subsample, colsample_bytree)
        (3,  0.05,  0.8, 0.8),
        (3,  0.1,   0.8, 0.8),
        (4,  0.05,  0.8, 0.8),
        (4,  0.1,   0.8, 0.8),
        (5,  0.05,  0.8, 0.8),
        (5,  0.1,   0.8, 0.8),
        (6,  0.05,  0.8, 0.8),
        (6,  0.1,   0.8, 0.8),
        (4,  0.05,  0.7, 0.7),
        (4,  0.1,   0.7, 0.7),
        (5,  0.05,  0.7, 0.7),
        (5,  0.1,   0.7, 0.7),
    ]

    # split train into train + internal validation for early stopping
    X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(
        X_train, y_train, sample_weights,
        test_size=0.15, random_state=42, stratify=y_train
    )

    best_f1      = -1.0
    best_model   = None
    best_params  = None
    results_log  = []

    print(f"\nSearching {len(param_candidates)} param combos with early stopping…")

    for i, (md, lr, ss, cs) in enumerate(param_candidates):
        model = xgb.XGBClassifier(
            objective          = 'multi:softprob',
            num_class          = n_classes,
            eval_metric        = 'mlogloss',
            max_depth          = md,
            learning_rate      = lr,
            n_estimators       = 1000,              # upper bound; early stopping will cut short
            subsample          = ss,
            colsample_bytree   = cs,
            reg_alpha          = 0.1,               # L1 regularisation
            reg_lambda         = 1.0,               # L2 regularisation
            random_state       = 42,
            use_label_encoder  = False,
            tree_method        = 'hist',            # fast CPU training
            early_stopping_rounds = 20
        )

        model.fit(
            X_tr, y_tr,
            sample_weight  = w_tr,
            eval_set       = [(X_val, y_val)],
            verbose        = False
        )

        # evaluate on the held-out TEST set with f1_macro
        from sklearn.metrics import f1_score
        y_pred_test = model.predict(X_test)
        f1 = f1_score(y_test, y_pred_test, average='macro')

        results_log.append({
            'max_depth': md, 'lr': lr, 'subsample': ss,
            'colsample': cs, 'best_iteration': model.best_iteration,
            'f1_macro_test': f1
        })

        if f1 > best_f1:
            best_f1     = f1
            best_model  = model
            best_params = {'max_depth': md, 'learning_rate': lr,
                           'subsample': ss, 'colsample_bytree': cs}

        print(f"  [{i+1:2d}/{len(param_candidates)}]  "
              f"depth={md} lr={lr} sub={ss} col={cs}  "
              f"→ best_iter={model.best_iteration:4d}  f1_macro={f1:.4f}"
              + ("  ★" if model is best_model else ""))

    # ── retrain best config on FULL training set (no val split) ──
    print(f"\nRetraining best config on full train set: {best_params}")
    final_model = xgb.XGBClassifier(
        objective          = 'multi:softprob',
        num_class          = n_classes,
        eval_metric        = 'mlogloss',
        max_depth          = best_params['max_depth'],
        learning_rate      = best_params['learning_rate'],
        n_estimators       = best_model.best_iteration + 1,   # use the iteration count we found
        subsample          = best_params['subsample'],
        colsample_bytree   = best_params['colsample_bytree'],
        reg_alpha          = 0.1,
        reg_lambda         = 1.0,
        random_state       = 42,
        use_label_encoder  = False,
        tree_method        = 'hist'
    )
    full_weights = compute_sample_weights(y_train)
    final_model.fit(X_train, y_train, sample_weight=full_weights, verbose=False)

    # ── final scores ──
    y_pred_train = final_model.predict(X_train)
    y_pred_test  = final_model.predict(X_test)

    print("-" * 70)
    print(f"Best params:       {best_params}")
    print(f"Best iteration:    {best_model.best_iteration}")
    print(f"Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test accuracy:     {accuracy_score(y_test,  y_pred_test):.4f}")

    # ── feature importance (gain-based, same as RF's impurity-based) ──
    importance = pd.DataFrame({
        'feature':    feature_cols,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 15 feature importances (gain):")
    print(importance.head(15).to_string(index=False))

    # ── search summary ──
    print("\n" + "-" * 70)
    print("All candidate results:")
    print(pd.DataFrame(results_log).sort_values('f1_macro_test', ascending=False).to_string(index=False))

    return final_model, importance


# ============================================================================
# 6. Evaluation
# ============================================================================

def evaluate_models(models_dict, X_test, y_test, label_encoder):
    """Classification report + confusion matrix for every model."""
    print("\n" + "=" * 70)
    print("Comprehensive model evaluation")
    print("=" * 70)

    for name, info in models_dict.items():
        print(f"\n{'=' * 70}\n{name.upper()}\n{'=' * 70}")

        model = info['model']
        X_proc = info['scaler'].transform(X_test) if 'scaler' in info else X_test
        y_pred = model.predict(X_proc)

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred,
                                    target_names=label_encoder.classes_, digits=4))

        print("Confusion Matrix:")
        cm = pd.DataFrame(confusion_matrix(y_test, y_pred),
                          index=label_encoder.classes_,
                          columns=label_encoder.classes_)
        print(cm)
        print(f"\nOverall Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")


# ============================================================================
# 7. Head-to-head comparison table
# ============================================================================

def print_comparison(models_dict, X_test, y_test, label_encoder):
    """Print a clean side-by-side accuracy / f1 comparison."""
    from sklearn.metrics import f1_score

    print("\n" + "=" * 70)
    print("MODEL COMPARISON")
    print("=" * 70)

    rows = []
    for name, info in models_dict.items():
        model  = info['model']
        X_proc = info['scaler'].transform(X_test) if 'scaler' in info else X_test
        y_pred = model.predict(X_proc)

        acc_macro = accuracy_score(y_test, y_pred)
        f1_macro  = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')

        # per-class f1
        f1_per = f1_score(y_test, y_pred, average=None)
        row = {'Model': name, 'Accuracy': acc_macro,
               'F1 Macro': f1_macro, 'F1 Weighted': f1_weighted}
        for cls, f1v in zip(label_encoder.classes_, f1_per):
            row[f'F1 {cls}'] = f1v
        rows.append(row)

    comp_df = pd.DataFrame(rows).set_index('Model').T
    print(comp_df.round(4).to_string())


# ============================================================================
# 8. Main
# ============================================================================

def main():
    print("=" * 70)
    print("Household income prediction  —  SVM + XGBoost")
    print("=" * 70)

    # 1. load
    trace_df, survey_df = load_data()

    # 2. survey labels
    survey_df = preprocess_survey_data(survey_df)

    # 3. datetime (single pass)
    trace_df['started_at']  = pd.to_datetime(trace_df['started_at'],  errors='coerce')
    trace_df['finished_at'] = pd.to_datetime(trace_df['finished_at'], errors='coerce')
    trace_df = trace_df.dropna(subset=['started_at', 'finished_at'])

    # 4. features
    features_df = build_features_per_user(trace_df)

    # 5. split
    X_train, X_test, y_train, y_test, label_encoder, feature_cols = \
        prepare_train_test_data(features_df, survey_df)

    # 6. SVM
    svm_model, svm_scaler = train_svm_model(X_train, y_train, X_test, y_test)

    # 7. XGBoost
    xgb_model, xgb_importance = train_xgboost_model(
        X_train, y_train, X_test, y_test, feature_cols)

    # 8. evaluate
    models_dict = {
        'SVM':     {'model': svm_model,  'scaler': svm_scaler},
        'XGBoost': {'model': xgb_model}
    }
    evaluate_models(models_dict, X_test, y_test, label_encoder)

    # comparison table
    print_comparison(models_dict, X_test, y_test, label_encoder)

    # save
    xgb_importance.to_csv('xfeature_importance_xgb.csv', index=False)

    import pickle
    with open('/data/baliu/python_code/data/svm_model.pkl', 'wb') as f:
        pickle.dump({'model': svm_model, 'scaler': svm_scaler}, f)
    with open('/data/baliu/python_code/data/xgb_model.pkl', 'wb') as f:
        pickle.dump(xgb_model, f)
    with open('/data/baliu/python_code/data/label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)

    print("\n✓ Models and feature importance saved successfully")


if __name__ == "__main__":
    main()

Household income prediction  —  SVM + XGBoost


Trace data shape:  (60738, 23)
Survey data shape: (90909, 124)

Preprocessing survey data...
Income category distribution:
income_category
0_<4000          2618
1_4001-8000      6454
2_8001-12000     5285
3_12001-16000    2575
4_>16001         1700
Name: count, dtype: int64

Building features from mobility traces...
Feature matrix shape: (2102, 31)
Number of features:   30

Preparing training and test data...
Merged shape: (1527, 34)   |   matched users: 1527

Feature columns (30):
['total_trips', 'morning_peak_trips', 'evening_peak_trips', 'peak_trip_ratio', 'weekday_trips', 'weekend_trips', 'weekend_trip_ratio', 'unique_hours', 'unique_days', 'unique_locations'] ...
Train: 1221   Test: 306

Training SVM model  (RBF kernel)

Grid search (scoring = f1_macro, class_weight = balanced)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best params:      {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV f1_macro: 0.2285
--------------------------------------------------------

## 3. Predict Sociodemographics using "gpt-oss-20b" model based on the prompts we just generated 

In [40]:
import ollama

models = ollama.list()
print(models)

models=[Model(model='gpt-oss:20b', modified_at=datetime.datetime(2025, 11, 9, 20, 40, 9, 692297, tzinfo=TzInfo(3600)), digest='17052f91a42e97930aa6e28a6c6c06a983e6a58dbb00434885a0cf5313e376f7', size=13793441244, details=ModelDetails(parent_model='', format='gguf', family='gptoss', families=['gptoss'], parameter_size='20.9B', quantization_level='MXFP4'))]


In [41]:
import os, json, re, subprocess, sys
import torch, requests
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory/1024**3, 2))

# Check ollama is reachable
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
print("Ollama OK. Models:", [m["name"] for m in r.json().get("models",[])])

CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.06
Ollama OK. Models: ['gpt-oss:20b']


read prompts from text version

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import ollama
import pandas as pd

# --------------------------------------
# Config
# --------------------------------------
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
client = ollama.Client(host=os.environ["OLLAMA_HOST"])


# prompts_v3_1week_income_29Jan2026_clean  prompts_v3_1week_income_29Jan2026
PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026_clean.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.csv")

MODEL_NAME = "gpt-oss:20b"
SEP = "=" * 80

# ---- safety knobs ----
MAX_PROMPT_CHARS = 20_000     # prompt length guard
TIMEOUT_SEC      = 600        # generation timeout per user
COOLDOWN_EVERY   = 20         # periodic cooldown interval
COOLDOWN_SEC     = 10

# --------------------------------------
# Timeout handling (linux)
# --------------------------------------
class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

# --------------------------------------
# Load checkpoint once
# --------------------------------------
done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

preds = []

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id, skipping")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        # ---- prompt length guard ----
        if len(prompt) > MAX_PROMPT_CHARS:
            print(f"⚠️ Prompt too long ({len(prompt)} chars), skip {user_id}")
            continue

        try:
            print("💓 Heartbeat: sending to model")
            start_t = time.time()

            signal.alarm(TIMEOUT_SEC)

            resp = client.generate(
                model=MODEL_NAME,
                prompt=prompt,
                options={
                    "temperature": 0.1,
                    "num_ctx": 2048
                }
            )

            signal.alarm(0)

            dt = time.time() - start_t
            print(f"⏱️ Model returned in {dt:.1f}s")

            content = resp.get("response", "").strip()

            try:
                result = json.loads(content)
            except Exception:
                result = {
                    "raw_output": content,
                    "error": "json_parse_failed"
                }

            result["user_id"] = user_id
            preds.append(result)

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())

            done_users.add(user_id)
            print(f"✅ Completed {user_id}")

        except TimeoutError:
            print(f"⏰ Timeout for {user_id}, skipping")
            continue

        except Exception as e:
            print(f" Error for {user_id}: {e}")
            time.sleep(2)
            continue            

        # ---- periodic cooldown ----
        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU / Ollama...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted by user. Progress saved safely.")
    sys.exit(0)

# --------------------------------------
# Save CSV snapshot
# --------------------------------------
if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved {len(preds)} new predictions → {PRED_CSV}")
else:
    print("⚠️ No new predictions generated")


📦 Loaded 2102 prompts
🔁 Found 692 completed users
⏭️ Skipping AAGAF
⏭️ Skipping AAINS
⏭️ Skipping AAQME
⏭️ Skipping AARWP
⏭️ Skipping ABARC
⏭️ Skipping ABECR
⏭️ Skipping ABENL
⏭️ Skipping ABISR
⏭️ Skipping ABLPH
⏭️ Skipping ACSUF
⏭️ Skipping ACWVM
⏭️ Skipping ADCMH
⏭️ Skipping AEBWY
⏭️ Skipping AEGHJ
⏭️ Skipping AEWVV
⏭️ Skipping AFNCT
⏭️ Skipping AFUVG
⏭️ Skipping AGHUY
⏭️ Skipping AHTYK
⏭️ Skipping AJJJR
⏭️ Skipping AJJNX
⏭️ Skipping AJKBY
⏭️ Skipping AJQGV
⏭️ Skipping AKGPG
⏭️ Skipping AKMIL
⏭️ Skipping AKOZE
⏭️ Skipping AKQKS
⏭️ Skipping ALKPX
⏭️ Skipping ALSMT
⏭️ Skipping AMGUM
⏭️ Skipping AMGZL
⏭️ Skipping AMYMD
⏭️ Skipping ANEFF
⏭️ Skipping ANMKC
⏭️ Skipping ANQHI
⏭️ Skipping AOVGU
⏭️ Skipping AOVLF
⏭️ Skipping APKPN
⏭️ Skipping APWYR
⏭️ Skipping AQFEA
⏭️ Skipping AQQGE
⏭️ Skipping AQREK
⏭️ Skipping ARFNG
⏭️ Skipping ARMZF
⏭️ Skipping ARZDI
⏭️ Skipping ASUKL
⏭️ Skipping ASXOD
⏭️ Skipping ATBSI
⏭️ Skipping ATCBP
⏭️ Skipping ATEAN
⏭️ Skipping ATKDY
⏭️ Skipping AUXLS
⏭️ Skipping AW

SystemExit: 0

/home/baliu/.venvs/gptoss-env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 

## Evaluation

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.jsonl")

df = pd.read_json(raw_path, lines=True)

print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income extraction functions
# --------------------------------------
def extract_from_text(text):
    if pd.isna(text):
        return None

    try:
        data = json.loads(text)
        income_level = data.get("income_level", "")
        if income_level:
            if income_level in [">=16000", ">=16001", "16001+", "16000+"]:
                return ">16000"
            return income_level
    except Exception:
        pass

    text = str(text).lower()

    norm = (
        text.lower()
            .replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    if re.search(
        r'(>=\s*16000|>=\s*16001|16001\+|16000\+|'
        r'more\s+than\s+16000|above\s+16000|at\s+least\s+16000)',
        norm
    ):
        return ">16000"

    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}[,.]?\d{3})\s*[-–]\s*(\d{1,2}[,.]?\d{3})', norm)

    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    else:
        return None

# --------------------------------------
# Apply extraction
# --------------------------------------
TEXT_COL = "raw_output"
df["income_level"] = df[TEXT_COL].apply(extract_from_text)

print(df.columns.tolist())
print(df.head(3))


print(df["income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned CSV
# --------------------------------------
#out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.jsonl")

df_clean = (
    df[["user_id", "income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.jsonl")
df_clean.to_json(out_path, orient="records", lines=True)
out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv")
df_clean.to_csv(out_path, index=False)


df_clean.to_csv(out_path, index=False)

print(f"✅ Clean predictions saved to: {out_path}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(20))


Loaded raw predictions: 414 rows
['raw_output', 'error', 'user_id', 'prediction', 'explanation', 'income_level']
                                          raw_output              error  \
0  The user’s mobility pattern shows a high level...  json_parse_failed   
1  The user’s mobility pattern shows a typical co...  json_parse_failed   
2  The user’s mobility pattern shows a typical su...  json_parse_failed   

  user_id prediction explanation income_level  
0   AAGAF        NaN         NaN  12001-16000  
1   AAINS        NaN         NaN    4001-8000  
2   AAQME        NaN         NaN   8001-12000  
income_level
8001-12000     192
12001-16000     82
>16000          47
None            47
4001-8000       37
<4000            9
Name: count, dtype: int64
✅ Clean predictions saved to: /data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv
📊 Rows: 414
   user_id income_level
0    AAGAF  12001-16000
1    AAINS    4001-8000
2    AAQME   8001-12000
3    AARWP  12001-16000
4    ABA

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv")

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]]
# pred = pred[["user_id", "age_group", "household_sizes", "income_level"]]
pred = pred[["user_id", "income_level"]]

# --------------------------------------
# Normalize household size
# --------------------------------------
# def normalize_household_size(x):
#     if pd.isna(x):
#         return np.nan
#     try:
#         x = float(x)
#     except Exception:
#         return np.nan

#     if x >= 5:
#         return "5+"
#     else:
#         return str(int(x))

# gt["hh_norm"] = gt["household_size_group"].apply(normalize_household_size)
# pred["hh_norm"] = pred["household_sizes"].apply(normalize_household_size)

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(
    gt,
    pred,
    on="user_id",
    suffixes=("_true", "_pred")
)

print(f"Merged {len(merged)} users")

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
# age_mask = (
#     merged["age_group_true"].notna() &
#     merged["age_group_pred"].notna()
# )

# hh_mask = (
#     merged["hh_norm_true"].notna() &
#     merged["hh_norm_pred"].notna()
# )

income_mask = (
    merged["income_level_true"].notna() &
    merged["income_level_pred"].notna() &
    (merged["income_level_true"].str.lower() != "prefer not to say")
)

print("\nEvaluation sample sizes:")
# print(f"Age:        {age_mask.sum()}")
# print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# # --------------------------------------
# age_acc = (
#     merged.loc[age_mask, "age_group_true"]
#     == merged.loc[age_mask, "age_group_pred"]
# ).mean()

# household_acc = (
#     merged.loc[hh_mask, "hh_norm_true"]
#     == merged.loc[hh_mask, "hh_norm_pred"]
# ).mean()

income_acc = (
    merged.loc[income_mask, "income_level_true"]
    == merged.loc[income_mask, "income_level_pred"]
).mean()

# --------------------------------------
# Majority-class baselines (for context)
# --------------------------------------
def majority_baseline(y_true):
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

# age_baseline = majority_baseline(
#     merged.loc[age_mask, "age_group_true"]
# )

# hh_baseline = majority_baseline(
#     merged.loc[hh_mask, "hh_norm_true"]
#)

# income_baseline = majority_baseline(
#     merged.loc[income_mask, "income_level_true"]
# )

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
# print(f"Age group accuracy:        {age_acc:.3f} (baseline {age_baseline:.3f})")
# print(f"Household size accuracy:  {household_acc:.3f} (baseline {hh_baseline:.3f})")
#print(f"Income level accuracy:    {income_acc:.3f} (baseline {income_baseline:.3f})")
print(f"Income level accuracy:    {income_acc:.3f} ")
# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v5_final.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path(
    "/data/baliu/python_code/data/comparison_income_eval_v5_final.csv"
)
merged.loc[income_mask].to_csv(
    filtered_income_path, index=False, encoding="utf-8"
)

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# (Optional) Confusion matrices
# --------------------------------------
# age_cm = pd.crosstab(
#     merged.loc[age_mask, "age_group_true"],
#     merged.loc[age_mask, "age_group_pred"],
#     normalize="index"
# )

# age_cm_path = Path(
#     "/data/baliu/python_code/data/age_confusion_matrix_v4_final.csv"
# )
# age_cm.to_csv(age_cm_path)

# print(f"Age confusion matrix saved to: {age_cm_path}")

# ---- Confusion matrix (income) ----
income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level_true"],
    merged.loc[income_mask, "income_level_pred"]
    #normalize="index"
)

print("\nIncome confusion matrix (counts):")
print(income_cm)

income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level_true"],
    merged.loc[income_mask, "income_level_pred"],
    normalize="index"
)

print("\nIncome confusion matrix (row-normalized):")
print(income_cm)

/tmp/ipykernel_930041/1873157119.py:14: DtypeWarning: Columns (4,6,7,8,9,10,11,12,13,15,17,18,19,20,21,22,23,28,29,30,31,32,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,113,114,115,116,117,118,119,120,121,122,123) have mixed types. Specify dtype option on import or set low_memory=False.
  gt = pd.read_csv(gt_path)


Loaded ground truth (90909 rows)
Loaded predictions  (414 rows)
Merged 414 users

Evaluation sample sizes:
Income:     328

📊 Accuracy Results:
Income level accuracy:    0.271 

Detailed comparison saved to: /data/baliu/python_code/data/merged_comparison_v5_final.csv
Income-eval subset saved to: /data/baliu/python_code/data/comparison_income_eval_v5_final.csv

Income confusion matrix (counts):
income_level_pred  12001-16000  4001-8000  8001-12000  <4000  >16000
income_level_true                                                   
12001-16000                 15          6          25      0       4
4001-8000                   23         15          51      2      19
8001-12000                  17          8          54      5      13
<4000                        4          3          21      0       3
>16000                      12          1          21      1       5

Income confusion matrix (row-normalized):
income_level_pred  12001-16000  4001-8000  8001-12000     <4000    >16000
inc